# Baseline Feature Retrieval, Missing Values & Outlier Handling

**Issue #14 — Address Missing Values & Outliers**

This notebook:
1. Loads baseline features (demographics, job values, job experience) from the ISSP constructed dataset — US respondents only
2. Merges in O\*NET occupation-level features (job zone, RIASEC interest profile) using the `ONET_SOC_CODE` join key
3. Handles missing values with appropriate strategies per column
4. Detects and handles outliers in continuous features
5. Saves the clean, merged dataset for downstream modeling

---

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../../data/'
CONSTRUCTED_DIR = DATA_DIR + 'constructed_datasets/'
ONET_DIR = DATA_DIR + 'formatted_onet/'

# ISSP country code for the United States (ISO 3166-1 numeric)
US_COUNTRY_CODE = 840

## 2. Load ISSP — US Respondents Only

In [ ]:
issp_full = pd.read_csv(CONSTRUCTED_DIR + 'issp.csv')

# Filter to US respondents only (country == 840).
# Our goal is a US-focused model, and pooling international respondents would
# introduce cultural differences in how job satisfaction is interpreted and
# reported, which would add noise rather than signal for a US-targeted prediction.
issp_us = issp_full[issp_full['country'] == US_COUNTRY_CODE].copy()

print(f'Total ISSP rows:    {len(issp_full):,}')
print(f'US respondents:     {len(issp_us):,}')
print(f'Survey waves:       {sorted(issp_us["module_year"].dropna().unique().astype(int))}')

## 3. Select Baseline Features

In [ ]:
# We select four groups of columns for the baseline model:
#   (a) Background / demographic factors — who the person is
#   (b) Job values (imp_*) — what the person wants from work
#   (c) Job experience (actual_*) — what their current job delivers
#   (d) Target and join keys
# This covers the core "person" side of person-job fit;
# occupation-level features from O*NET are joined in the next section.

BACKGROUND_COLS = [
    'age',                      # continuous — respondent age in years
    'sex',                      # binary (1 = male, 2 = female)
    'marital_status',           # categorical (1 = married … 5 = never married)
    'educ_years',               # continuous — years of education completed
    'educ_level',               # ordinal (0 = no qual … 5 = university+)
    'household_income_group',   # ordinal (1 = low, 2 = middle, 3 = high)
    'hours_worked_weekly',      # continuous — paid work hours per week
    'employment_status_current',# categorical (1 = full-time … 8 = other)
    'union_member',             # categorical (1 = yes, 2 = no, 3 = formerly)
    'household_size',           # continuous — number of people in household
    'child_in_household',       # binary (1 = yes, 2 = no)
    'urban_rural',              # ordinal (1 = urban … 5 = rural)
]

JOB_VALUE_COLS = [
    'imp_job_security',    # How important is job security to you? (1–5)
    'imp_income',          # High income?
    'imp_advancement',     # Opportunities for advancement?
    'imp_interesting',     # Interesting work?
    'imp_independence',    # Work independently?
    'imp_help_others',     # Help others?
    'imp_useful_society',  # Be useful to society?
    'imp_decide_hours',    # Decide your own working hours?
]

JOB_ACTUAL_COLS = [
    'actual_job_secure',          # Is your job actually secure? (1–5)
    'actual_income_high',         # Is the income actually high?
    'actual_advancement_high',    # Are there advancement opportunities?
    'actual_job_interesting',     # Is the work actually interesting?
    'actual_work_independently',  # Do you actually work independently?
    'actual_help_others',         # Does the job let you help others?
    'actual_useful_society',      # Is the work useful to society?
]

TARGET_COL = 'job_satisfaction'   # 1 (completely satisfied) – 7 (completely dissatisfied)
JOIN_KEY   = 'ONET_SOC_CODE'
WAVE_COL   = 'module_year'

ALL_ISSP_COLS = [JOIN_KEY, WAVE_COL, TARGET_COL] + BACKGROUND_COLS + JOB_VALUE_COLS + JOB_ACTUAL_COLS

df = issp_us[ALL_ISSP_COLS].copy()
print(f'Selected columns: {df.shape[1]}')
print(f'Rows: {df.shape[0]:,}')
df.head(3)

## 4. Load O\*NET Features & Merge

In [ ]:
# Load O*NET job zones (occupation complexity level, 1–5).
# Job zone is a compact, single-number summary of how much preparation
# an occupation typically requires; it captures education and training
# demands without needing to bring in all 200 O*NET feature columns.
onet_jobzones = pd.read_csv(ONET_DIR + 'formatted_jobzones.csv')[['ONET_SOC_CODE', 'Job Zone']]
onet_jobzones = onet_jobzones.rename(columns={'Job Zone': 'onet_job_zone'})

# Load O*NET RIASEC interest scores (6 Holland interest types).
# RIASEC scores characterise the social and intellectual texture of each
# occupation (e.g. high Social score = lots of human interaction), which maps
# directly onto many of the ISSP job-value dimensions we care about.
onet_interests = pd.read_csv(ONET_DIR + 'formatted_career_interest_types.csv')
onet_interests = onet_interests.rename(columns={'O*NET-SOC Code': 'ONET_SOC_CODE'})

RIASEC_COLS = [
    'career_interest_types_OI_artistic_value',
    'career_interest_types_OI_conventional_value',
    'career_interest_types_OI_enterprising_value',
    'career_interest_types_OI_investigative_value',
    'career_interest_types_OI_realistic_value',
    'career_interest_types_OI_social_value',
]
onet_interests = onet_interests[['ONET_SOC_CODE'] + RIASEC_COLS]

print(f'O*NET job zones:  {onet_jobzones.shape[0]} occupations')
print(f'O*NET interests:  {onet_interests.shape[0]} occupations')

In [ ]:
# Merge ISSP with O*NET using a left join on ONET_SOC_CODE.
# A left join preserves all ISSP respondents; rows without an O*NET match
# receive NaN for occupation features, which will be handled in the
# missing-value section below.
df = df.merge(onet_jobzones, on='ONET_SOC_CODE', how='left')
df = df.merge(onet_interests, on='ONET_SOC_CODE', how='left')

ONET_COLS = ['onet_job_zone'] + RIASEC_COLS

print(f'Merged dataset shape: {df.shape}')
print(f'\nO*NET match rate:')
print(f'  Job zone matched:     {df["onet_job_zone"].notna().sum():,} / {len(df):,} '
      f'({df["onet_job_zone"].notna().mean()*100:.1f}%)')
print(f'  RIASEC matched:       {df[RIASEC_COLS[0]].notna().sum():,} / {len(df):,} '
      f'({df[RIASEC_COLS[0]].notna().mean()*100:.1f}%)')

## 5. Missing Value Audit

In [ ]:
# Produce a complete missingness summary before any imputation so that
# we can document the baseline null rates and justify each strategy.
miss = df.isnull().sum().rename('n_missing')
miss = miss.to_frame()
miss['pct_missing'] = (miss['n_missing'] / len(df) * 100).round(1)
miss = miss[miss['n_missing'] > 0].sort_values('pct_missing', ascending=False)

print(f'Columns with missing values: {len(miss)} / {df.shape[1]}')
print()
print(miss.to_string())

In [ ]:
# Visualise missingness as a horizontal bar chart for easy reading.
fig, ax = plt.subplots(figsize=(8, max(4, len(miss) * 0.35)))
colors = ['#e74c3c' if p > 40 else '#e67e22' if p > 15 else '#3498db'
          for p in miss['pct_missing']]
ax.barh(miss.index, miss['pct_missing'], color=colors)
ax.axvline(x=40, color='red', linestyle='--', linewidth=0.8, label='40% threshold')
ax.set_xlabel('% Missing')
ax.set_title('Missing Values by Feature (US ISSP + O*NET Merged)')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Handle Missing Values

We apply different strategies depending on **why** data is missing and **how much** is missing.

### 6a. Target variable — `job_satisfaction`

Rows missing the target cannot be used for supervised learning. We drop them rather than imputing, because fabricating a satisfaction score would introduce false labels that directly corrupt model training.

In [ ]:
n_before = len(df)
df = df[df['job_satisfaction'].notna()].copy()
n_after = len(df)
print(f'Dropped {n_before - n_after:,} rows with missing job_satisfaction.')
print(f'Remaining rows: {n_after:,}')

### 6b. `actual_*` columns — missing because respondent was not currently employed

The `actual_*` questions were only asked of currently employed workers, so their missingness is structurally tied to employment status rather than random non-response. We drop rows missing these columns because without them we cannot compute the person-job fit gaps that are central to our model.

In [ ]:
n_before = len(df)
df = df.dropna(subset=JOB_ACTUAL_COLS).copy()
n_after = len(df)
print(f'Dropped {n_before - n_after:,} rows missing actual_* experience columns.')
print(f'Remaining rows: {n_after:,}')

### 6c. `imp_*` columns — small random missingness (~2%), median imputation

Importance columns have less than 2.5% missing values, which is low enough that imputation introduces negligible bias. We use the **median** rather than the mean because the 1–5 Likert scale is ordinal, and the median respects that ordering without being pulled by extreme values.

In [ ]:
for col in JOB_VALUE_COLS:
    n_missing = df[col].isnull().sum()
    if n_missing > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f'  {col}: filled {n_missing} NaN → median ({median_val:.1f})')

print('\nimp_* missing after imputation:', df[JOB_VALUE_COLS].isnull().sum().sum())

### 6d. `hours_worked_weekly` — median imputation within employment-status group

Hours worked is missing for ~36% of respondents (primarily those not currently employed full-time). We impute using the **median within each employment status group** rather than the global median, because part-time and full-time workers have very different expected hours; group-wise imputation better preserves this variation.

In [ ]:
n_missing_hours = df['hours_worked_weekly'].isnull().sum()

group_medians = df.groupby('employment_status_current')['hours_worked_weekly'].transform('median')
global_median = df['hours_worked_weekly'].median()

# Fill with group median; fall back to global median if group is entirely NaN
df['hours_worked_weekly'] = df['hours_worked_weekly'].fillna(group_medians).fillna(global_median)

print(f'Imputed {n_missing_hours:,} missing hours_worked_weekly values using group medians.')
print(f'Global median fallback: {global_median:.1f} hrs')
print(f'Missing after imputation: {df["hours_worked_weekly"].isnull().sum()}')

### 6e. `household_income_group` — mode imputation

`household_income_group` is an ordinal categorical variable (Low / Middle / High) with ~10% missing values. We impute with the **mode** (most frequent category) because there is no meaningful numeric distance between income bands, making mean/median inappropriate for this column type.

In [ ]:
n_missing = df['household_income_group'].isnull().sum()
mode_val = df['household_income_group'].mode()[0]
df['household_income_group'] = df['household_income_group'].fillna(mode_val)
print(f'Imputed {n_missing} missing household_income_group → mode ({mode_val:.0f})')

### 6f. `union_member` — mode imputation

`union_member` is a categorical variable (yes / no / formerly) with ~16% missing values. Mode imputation is appropriate here for the same reason as `household_income_group` — it is a nominal category with no natural ordering, so average-based imputation would be meaningless.

In [ ]:
n_missing = df['union_member'].isnull().sum()
mode_val = df['union_member'].mode()[0]
df['union_member'] = df['union_member'].fillna(mode_val)
print(f'Imputed {n_missing} missing union_member → mode ({mode_val:.0f})')

### 6g. O\*NET features — drop rows with no occupation match

Rows missing O\*NET features have no valid `ONET_SOC_CODE` and therefore no occupation-level information at all. Since the O\*NET job profile is a core input in our model, these rows cannot contribute meaningful signal and are dropped rather than imputed.

In [ ]:
n_before = len(df)
df = df.dropna(subset=ONET_COLS).copy()
n_after = len(df)
print(f'Dropped {n_before - n_after:,} rows with no O*NET occupation match.')
print(f'Remaining rows: {n_after:,}')

### 6h. Remaining demographic columns — median/mode imputation

Columns like `age`, `educ_years`, `marital_status`, `educ_level`, `household_size`, `child_in_household`, and `urban_rural` have very low missingness (< 1%). For continuous/ordinal columns we use the median; for nominal categoricals we use the mode. At under 1% missingness, the choice of imputation strategy has negligible impact on model performance.

In [ ]:
# Continuous/ordinal columns → median
MEDIAN_COLS = ['age', 'educ_years', 'educ_level', 'household_size']
# Nominal categorical columns → mode
MODE_COLS = ['marital_status', 'child_in_household', 'urban_rural', 'employment_status_current']

for col in MEDIAN_COLS:
    n = df[col].isnull().sum()
    if n > 0:
        val = df[col].median()
        df[col] = df[col].fillna(val)
        print(f'  {col}: filled {n} NaN → median ({val:.1f})')

for col in MODE_COLS:
    n = df[col].isnull().sum()
    if n > 0:
        val = df[col].mode()[0]
        df[col] = df[col].fillna(val)
        print(f'  {col}: filled {n} NaN → mode ({val:.0f})')

print('\nTotal missing values remaining:', df.isnull().sum().sum())

### 6i. Missingness confirmation

In [ ]:
remaining_miss = df.isnull().sum()
remaining_miss = remaining_miss[remaining_miss > 0]

if len(remaining_miss) == 0:
    print('✅ No missing values remain in the dataset.')
else:
    print('⚠️  Columns still containing missing values:')
    print(remaining_miss)

## 7. Outlier Detection & Handling

We check outliers only in **continuous numeric** features. Ordinal Likert-scale variables (1–5 or 1–7) and coded categoricals are bounded by design, so they cannot contain true outliers — unusual values in those columns reflect real extreme opinions, not data errors.

### 7a. `hours_worked_weekly` — IQR-based Winsorisation

Hours per week ranges from 1 to 89 in the data. Values above 80 hours (40 observations) are plausible but extreme; they likely reflect self-employed individuals or survey rounding errors. We **Winsorise at the 1st and 99th percentiles** — capping rather than dropping — so extreme workers remain in the dataset but their influence on the model is reduced.

In [ ]:
col = 'hours_worked_weekly'
p01 = df[col].quantile(0.01)
p99 = df[col].quantile(0.99)

n_below = (df[col] < p01).sum()
n_above = (df[col] > p99).sum()

df[col] = df[col].clip(lower=p01, upper=p99)

print(f'{col}: 1st pct={p01:.0f} hrs, 99th pct={p99:.0f} hrs')
print(f'  Winsorised: {n_below} below lower cap, {n_above} above upper cap')
print(f'  New range: {df[col].min():.0f} – {df[col].max():.0f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
issp_us['hours_worked_weekly'].dropna().plot(kind='box', ax=axes[0], vert=False)
axes[0].set_title('Before Winsorisation')
df[col].plot(kind='box', ax=axes[1], vert=False)
axes[1].set_title('After Winsorisation')
plt.suptitle('hours_worked_weekly', fontsize=11)
plt.tight_layout()
plt.show()

### 7b. `age` — review only, no capping needed

Age ranges from 18 to 89, which is within the valid range for employed survey respondents. The upper tail reflects genuinely older workers rather than data errors, so no capping is applied — the IQR method would be overly aggressive on a naturally right-skewed demographic variable.

In [ ]:
col = 'age'
desc = df[col].describe()
print(f'{col} statistics:')
print(desc.to_string())
print(f'\nIQR fence (1.5×IQR): lower={desc["25%"] - 1.5*(desc["75%"]-desc["25%"]):.1f}, '
      f'upper={desc["75%"] + 1.5*(desc["75%"]-desc["25%"]):.1f}')
print('→ No extreme values outside valid 18–89 range. No action taken.')

### 7c. `educ_years` — review only, no capping needed

`educ_years` ranges from 0 to 20, which is entirely within the plausible range (0 = no formal education, 20 = advanced doctorate). No values exceed the theoretical maximum for US educational attainment, so no intervention is necessary.

In [ ]:
col = 'educ_years'
print(df[col].describe().to_string())
print('→ Range [0, 20] is valid. No action taken.')

### 7d. O\*NET RIASEC scores — review for plausibility

RIASEC values are O\*NET-reported scores on a 0–7 scale. Since these come from a controlled US government database rather than self-report, they should not contain erroneous outliers. We verify the range is within bounds as a sanity check.

In [ ]:
riasec_desc = df[RIASEC_COLS].describe().loc[['min', 'max']]
print('RIASEC score range check (expected 0–7):')
print(riasec_desc.to_string())

out_of_range = ((df[RIASEC_COLS] < 0) | (df[RIASEC_COLS] > 7)).any().any()
if out_of_range:
    print('⚠️  Out-of-range values detected — review needed.')
else:
    print('\n✅ All RIASEC values within expected 0–7 range. No action taken.')

## 8. Engineer Person-Job Fit Gap Features

The fit gap for each dimension is computed as `imp_X − actual_X`, capturing how far reality falls short of the person's preferences. A positive gap indicates that the job delivers less than what the person values; a negative gap indicates over-delivery. These gap features are the primary operationalisation of person-job fit in our model.

In [ ]:
# Matching pairs: (imp_col, actual_col, gap_name)
FIT_PAIRS = [
    ('imp_job_security',   'actual_job_secure',         'fit_gap_security'),
    ('imp_income',         'actual_income_high',         'fit_gap_income'),
    ('imp_advancement',    'actual_advancement_high',    'fit_gap_advancement'),
    ('imp_interesting',    'actual_job_interesting',     'fit_gap_interesting'),
    ('imp_independence',   'actual_work_independently',  'fit_gap_independence'),
    ('imp_help_others',    'actual_help_others',         'fit_gap_help_others'),
    ('imp_useful_society', 'actual_useful_society',      'fit_gap_useful_society'),
]

for imp_col, actual_col, gap_name in FIT_PAIRS:
    df[gap_name] = df[imp_col] - df[actual_col]

GAP_COLS = [g for _, _, g in FIT_PAIRS]
print('Fit gap features created:')
print(df[GAP_COLS].describe().round(2).to_string())

## 9. Final Dataset Summary

In [ ]:
print('=' * 55)
print('CLEAN MERGED DATASET SUMMARY')
print('=' * 55)
print(f'Total rows:              {len(df):,}')
print(f'Total columns:           {df.shape[1]}')
print(f'Remaining missing vals:  {df.isnull().sum().sum()}')
print()
print(f'Unique O*NET occupations: {df["ONET_SOC_CODE"].nunique()}')
print()
print('Survey wave breakdown:')
print(df['module_year'].value_counts().sort_index().to_string())
print()
print('Target (job_satisfaction) distribution:')
sat_dist = df['job_satisfaction'].value_counts().sort_index()
for v, c in sat_dist.items():
    bar = '█' * int(c / 10)
    print(f'  {v:.0f}: {bar} ({c:,})')
print()
print('Column groups:')
print(f'  Background demographics:     {len(BACKGROUND_COLS)}')
print(f'  Job value preferences (imp): {len(JOB_VALUE_COLS)}')
print(f'  Job experience (actual):     {len(JOB_ACTUAL_COLS)}')
print(f'  Person-job fit gaps:         {len(GAP_COLS)}')
print(f'  O*NET occupation features:   {len(ONET_COLS)}')

In [ ]:
# Plot the distribution of each fit gap to confirm they behave reasonably.
# We expect a slight positive skew (jobs often under-deliver on what people value).
fig, axes = plt.subplots(2, 4, figsize=(14, 5), sharey=False)
axes = axes.flatten()

for i, gap_col in enumerate(GAP_COLS):
    label = gap_col.replace('fit_gap_', '').replace('_', ' ').title()
    axes[i].hist(df[gap_col], bins=9, range=(-4, 4), color='#3498db', edgecolor='white', linewidth=0.5)
    axes[i].set_title(label, fontsize=9)
    axes[i].axvline(0, color='red', linestyle='--', linewidth=0.8)
    axes[i].set_xlabel('Gap (want − get)', fontsize=8)

# Hide the unused 8th subplot
axes[7].set_visible(False)

plt.suptitle('Person-Job Fit Gap Distributions (positive = job under-delivers)', fontsize=11)
plt.tight_layout()
plt.show()

## 10. Save Clean Dataset

In [ ]:
# Save to the constructed_datasets folder so downstream modeling notebooks
# can load a single, clean file without repeating the merge and cleaning steps.
# The filename includes 'baseline' to signal this is the feature-selected,
# cleaned starting point for model development.
OUT_PATH = CONSTRUCTED_DIR + 'issp_onet_baseline_us.csv'
df.to_csv(OUT_PATH, index=False)
print(f'✅ Saved clean dataset to: {OUT_PATH}')
print(f'   Shape: {df.shape}')